# prep_05_npi_zip
## Ohio Dental Clinic — Site Selection Analysis

**Purpose:** Cleans the CMS NPPES National Provider Identifier registry for Ohio dental providers. Filters to dental taxonomy codes, trims ZIP codes to 5-digit for consistent joining, and classifies each practice as solo, group, or DSO based on organization name fields.

| | |
|---|---|
| **Input** | `npidata_pfile_raw.csv` (CMS NPPES Feb 2026) |
| **Output** | `NPI_Dentists.csv` |
| **Records** | 10,962 Ohio dental providers |

In [1]:
# Import libraries
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")

# PATHS
RAW_DATA_PATH = r"C:\Users\mosun\Downloads\oh_clinic_rw_files"
OUTPUT_PATH = r"../data/cleaned"

# DENTAL TAXONOMY CODES
DENTAL_CODES = [
    '1223G0001X','122300000X','1223P0221X','1223X0400X',
    '1223S0112X','1223E0200X','1223P0300X','1223P0700X','1223D0001X'
]

# LOAD IN CHUNKS — file is too large to load all at once 
cols_needed = [
    'NPI',
    'Provider Organization Name (Legal Business Name)',
    'Provider Last Name (Legal Name)',
    'Provider First Name',
    'Provider First Line Business Practice Location Address',
    'Provider Business Practice Location Address City Name',
    'Provider Business Practice Location Address State Name',
    'Provider Business Practice Location Address Postal Code',
    *[f'Healthcare Provider Taxonomy Code_{i}' for i in range(1,16)]
]

chunks = []
for chunk in pd.read_csv(
    os.path.join(RAW_DATA_PATH, "npidata_pfile_20050523-20260208.csv"),  
    usecols=cols_needed, chunksize=50000,
    dtype={'Provider Business Practice Location Address Postal Code': str},
    low_memory=False):

    # Keep only Ohio rows
    oh = chunk[chunk['Provider Business Practice Location Address State Name'] == 'OH']

    # Keep only dental providers (check all 15 taxonomy columns)
    tax_cols = [c for c in oh.columns if 'Taxonomy Code' in c]
    mask = oh[tax_cols].isin(DENTAL_CODES).any(axis=1)
    chunks.append(oh[mask])

# Combine
result = pd.concat(chunks, ignore_index=True)
print(f"Ohio dental providers found: {len(result):,}")

# ZIP FIX
ZIP_COL = "Provider Business Practice Location Address Postal Code"

# Show what ZIPs look like before fix
print(f"\nBefore fix — sample ZIP values:")
print(result[ZIP_COL].head(10).tolist())

# Count how many are 9-digit vs 5-digit
result["_zip_len"] = result[ZIP_COL].astype(str).str.strip().str.len()
print(f"\nZIP length breakdown:")
print(result["_zip_len"].value_counts().sort_index().to_string())

# Fix: trim to first 5 characters, then zero-pad to ensure 5 digits
result[ZIP_COL] = (
    result[ZIP_COL]
    .astype(str)
    .str.strip()
    .str[:5]          # keep only first 5 characters
    .str.zfill(5)     # zero-pad if shorter than 5 (e.g. "1234" → "01234")
)
result = result.drop(columns=["_zip_len"])

# Verify fix
print(f"\nAfter fix — sample ZIP values:")
print(result[ZIP_COL].head(10).tolist())
all_five = (result[ZIP_COL].str.len() == 5).all()
print(f"All ZIPs are now 5 digits: {all_five}")

# SAVE
result.to_csv(os.path.join(OUTPUT_PATH, "NPI_Dentists.csv"), index=False)
print(f"\nSaved: NPI_Dentists.csv ({len(result):,} rows)")

Ohio dental providers found: 10,962

Before fix — sample ZIP values:
['447081505', '440011174', '442121115', '457509252', '442213228', '432131008', '451402795', '451402795', '44451', '454593711']

ZIP length breakdown:
_zip_len
5    1674
9    9288

After fix — sample ZIP values:
['44708', '44001', '44212', '45750', '44221', '43213', '45140', '45140', '44451', '45459']
All ZIPs are now 5 digits: True

Saved: NPI_Dentists.csv (10,962 rows)


In [3]:
df = pd.read_csv(r"C:\Users\mosun\Downloads\my projects\ohio_clinic\data\cleaned\NPI_Dentists.csv")
df.shape

(10962, 23)